In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
import json

In [ ]:
labels = []
for i in np.linspace(0.5, 4, 8):
    number = (str(i).split('.'))
    if number[-1] == '0':
        number = number[:-1]
    label = '_'.join(number)
    labels.append(label)

In [ ]:
label = 30

In [ ]:
with open('../data/ZigZag_3_4/ZigZag_3_4_iter_{}.json'.format(label), 'r') as f:
    data = json.load(f)

In [ ]:
fusedVertices = data['FusedVertices']

In [ ]:
V = data['Vertices']
F = data['Faces']

In [ ]:
m = MeshFEM.Mesh(V, F)

In [ ]:
fusedVertices == 1

In [ ]:

finalMarkers = np.where(np.array(fusedVertices) == 1)[0]
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
# m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

In [ ]:
pointList = np.where(np.array(fusedVtx) == 1)

In [ ]:
visualization.plot_2d_mesh(m, pointList=pointList, width=10, height=10)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)


In [ ]:

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + [ipu.numVars() - 2, ipu.numVars() - 1], 0
fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
# fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)

ipu.sheet.pressure = 0.01

In [ ]:
import numpy as np

In [ ]:
for test in [5, 5.5, 6, 8, 10, 12, 15, 20]:
    A = []
    for i in range(5):
        theta = np.pi / test * i
        A.append([np.cos(theta)**2 * np.sin(theta)**2, np.cos(theta) ** 3 * np.sin(theta), np.cos(theta) * np.sin(theta)**3, np.cos(theta) ** 4,  np.sin(theta)**4])
    print(np.linalg.cond(A))      

In [ ]:
A = []
for i in range(5):
    theta = np.pi / 5 * i
    A.append([np.cos(theta)**2 * np.sin(theta)**2, np.cos(theta) ** 3 * np.sin(theta), np.cos(theta) * np.sin(theta)**3, np.cos(theta) ** 4,  np.sin(theta)**4])
print(np.linalg.cond(A))      

In [ ]:
np.set_printoptions(precision=4)

In [ ]:
print(np.array(A))

In [ ]:
# ipu.sheet.disableFusedRegionTensionFieldTheory(False)

#### Problematic cell

In [ ]:
opts.ngd_fallback_steps = 6

In [ ]:
benchmark.reset()
opts.niter = 2000
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
        viewer.update()
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr.success
benchmark.report()

In [ ]:
benchmark.reset()
opts.niter = 2000
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        # viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
        viewer.update()
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr.success
benchmark.report()

In [ ]:
name = 'parallel_tube'
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  

az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx)

In [ ]:
az_ipu = inflation.InflatableMidSurfacePeriodicUnit(m, fusedVtx, epsilon = 1e-5)
az_ipu.ipu.setVars(ipu.getVars())
az_ipu.ipu.sheet.setUseTensionFieldEnergy(True)
az_ipu.ipu.sheet.setUseHessianProjectedEnergy(False)
az_ipu.ipu.sheet.pressure = ipu.sheet.pressure

In [ ]:
az_ipu.energy()

In [ ]:
np.linalg.norm(az_ipu.gradient())

In [ ]:

from tri_mesh_viewer import TriMeshViewer
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)

In [ ]:
az_viewer.show()

In [ ]:
az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
def az_cb(it):
    if it % framerate == 0:
        az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
opts.niter = 400
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
ipu.get_kappa()

In [ ]:
benchmark.reset()
stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = 1e-10, fixedVars = [], filename = "{}/stiffness_parallel_tube.png".format(result_folder))
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
benchmark.reset()
stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 100, az_optimizer, hessianShift = 1e-10, fixedVars = [], filename = "{}/stiffness_parallel_tube.png".format(result_folder), use_bases=False)
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
# curr_vars = az_ipu.getVars()
# curr_vars[-1] = np.pi / 2
# curr_vars[-2] = 0.5
# az_ipu.setVars(curr_vars)

In [ ]:
az_viewer.update()

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian(), reflect = True)

In [ ]:
H[-2, -2]

In [ ]:
bending_stiffness_sample_alpha = inflation.getBendingStiffness(az_ipu, [np.pi / 2], az_optimizer, 1e-10, [])



In [ ]:
bending_stiffness_sample_alpha

In [ ]:
min(stiffness_values), max(stiffness_values)

In [ ]:
points = visualize_average_deformation_gradient(ipu, 100, filename = "{}/average_deformation_gradient_parallel_tube.png".format(result_folder))

In [ ]:

an = np.linspace(0, 2 * np.pi, 100)
fig, ax = plt.subplots(1, 1)
ax.plot(np.cos(an), np.sin(an))

ax.plot(points[:, 0], points[:, 1])

ax.set_aspect('equal', 'box')
ax.set_title('still a circle, auto-adjusted data limits', fontsize=10)

fig.tight_layout()

plt.show()

In [ ]:

render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/render_merge_two_dash_{}.png".format(result_folder, label))

np.save("{}/stiffness_values_merge_two_dash_{}.npy".format(result_folder, label), stiffness_values)
np.save("{}/sampled_alphas_merge_two_dash_{}.npy".format(result_folder, label), sampled_alphas)
np.save("{}/scale_factors_merge_two_dash_{}.npy".format(result_folder, label), get_deformation_scale_factors(ipu))

In [ ]:
# stiffness_values, sampled_alphas, 
get_deformation_scale_factors(ipu)

### Inflatable experiment

In [ ]:
# for i in range(2):
#     m, markers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, markers, axis = 0)
#     m, markers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, markers, axis = 0)

In [ ]:
fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), markers)

In [ ]:
fuse_boundary = True

In [ ]:
# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            fusedVtx[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            fusedVtx[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            fusedVtx[i] = True    

In [ ]:
isheet = inflation.InflatableSheet(m, fusedVtx = fusedVtx)

from tri_mesh_viewer import TriMeshViewer
sheet_viewer = TriMeshViewer(isheet, width=768, height=640)
sheet_viewer.showWireframe(True)

In [ ]:
sheet_viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion

fixedVars, hessianShift = [], 1e-8

isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.disableFusedRegionTensionFieldTheory(False)

isheet.pressure = 1


opts.niter = 500
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        sheet_viewer.update(scalarField=utils.getStrains(isheet)[:, 0])
cr = inflation.inflation_newton(isheet, fixedVars, opts, callback=cb, hessianShift = hessianShift)